# APIM ❤️ Microsoft Foundry

## AI Foundry Hosted Agents lab

This lab demonstrates how to build and deploy an **AI Foundry Hosted Agent** to Azure Container Apps (ACA).
The agent is backed by two AI Foundry projects:
- **foundry-inference** (Sweden Central): hosts the `gpt-4.1` model deployment, accessed through APIM.
- **foundry-agent** (East US 2): serves as the orchestration layer for the hosted agent with a Weather MCP connection.

### Architecture
![flow](../../images/ai-foundry-hosted-agents.gif)

**Why this architecture matters?**

This architecture resembles a federated approach where the models and the tools are managed centrally by a platform team, where the AI Apps team would be responsible for the development of their agents.

The Models foundry is solely hosting Foundry models for inference, and some APIs are secured behind APIM.

The Agents foundry connects to the models and the tools (such as MCP, Bing Search, etc) via Foundry tools connections through APIM

The hosted agent is hosted on ACA and uses the connections available in the Agents Foundry without having to hold any credentials for the backend components, only via RBAC for the Agents Foundry.

The hosted agents could be built using any framework such Agent Framework, Langraph, or anytging cutom-built. This allows organisations to build a hybrid architecture that promotes interoprability and bringing all agents into the same Foundry Control plane.

- [Hosted Agents](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents?view=foundry)
- [Hosted Agents Samples](https://github.com/microsoft-foundry/foundry-samples/tree/main/samples/python/hosted-agents)
- [Azure AI Agent Server Samples](https://github.com/Azure/azure-sdk-for-python/tree/main/sdk/agentserver/azure-ai-agentserver-agentframework/samples)

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [Python environment](https://code.visualstudio.com/docs/python/environments#_creating-environments) with the [requirements.txt](../../requirements.txt) or run `pip install -r requirements.txt` in your terminal
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)
- [Docker](https://www.docker.com/get-started/) installed and running

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`...


<a id='0'></a>
### 0️⃣ Initialize notebook variables

Configure resource names, regions, model deployments, and API paths used throughout this lab.

In [ ]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}"
resource_group_location = 'westeurope'

# AI Services configuration
# foundry-inference: Sweden Central – receives the gpt-4.1 model deployment
# foundry-agent:     East US 2     – used as the hosted agent project (no direct model)
aiservices_config = [
    {"name": "foundry-inference", "location": "swedencentral", "weight": 1},
    {"name": "foundry-agent",     "location": "eastus2", "weight": 0},
]

# 'aiservice' field restricts model deployment to foundry-inference only
models_config = [
    {"name": "gpt-4.1-mini", "publisher": "OpenAI", "version": "2025-04-14",
     "sku": "GlobalStandard", "capacity": 50, "aiservice": "foundry-inference"}
]

# APIM configuration
apim_sku = 'Basicv2'
apim_subscriptions_config = [{"name": "subscription1", "displayName": "Subscription 1"}]

# API paths
inference_api_path = 'inference'
inference_api_type = 'PassThrough'
weather_mcp_path = 'weather'
foundry_project_name = deployment_name

build=0

utils.print_ok('Notebook initialized')

<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

The following commands ensure that you have the latest version of the Azure CLI and that the Azure CLI is connected to your Azure subscription.

In [ ]:
output = utils.run('az account show', 'Retrieved az account', 'Failed to get the current az account')

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']
    utils.print_info(f'Current user: {current_user}')
    utils.print_info(f'Tenant ID:    {tenant_id}')
    utils.print_info(f'Subscription: {subscription_id}')

<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declaratively define all resources deployed in the resource group.

Deployed resources include:
- Log Analytics Workspace + Application Insights
- Azure API Management (Basicv2)
- Two AI Foundry accounts (`foundry-inference` with gpt-4.1, `foundry-agent` for hosted agent)
- Container Registry + Container App Environment
- Weather MCP Server ACA (placeholder image)
- Hosted Agent ACA (placeholder image)
- APIM Streamable MCP API for the Weather server
- Weather MCP Foundry connection


In [ ]:
# Create the resource group
utils.create_resource_group(resource_group_name, resource_group_location)

# Build the Bicep parameter file
bicep_parameters = {
    '$schema': 'https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#',
    'contentVersion': '1.0.0.0',
    'parameters': {
        'apimSku':                  {'value': apim_sku},
        'aiServicesConfig':         {'value': aiservices_config},
        'modelsConfig':             {'value': models_config},
        'apimSubscriptionsConfig':  {'value': apim_subscriptions_config},
        'inferenceAPIPath':         {'value': inference_api_path},
        'inferenceAPIType':         {'value': inference_api_type},
        'weatherMCPPath':           {'value': weather_mcp_path},
        'foundryProjectName':       {'value': foundry_project_name},
    }
}

with open('params.json', 'w') as f:
    f.write(json.dumps(bicep_parameters))

output = utils.run(
    f'az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json',
    f"Deployment '{deployment_name}' succeeded",
    f"Deployment '{deployment_name}' failed"
)

<a id='3'></a>
### 3️⃣ Get the deployment outputs

Retrieve the gateway URL, subscription keys, Foundry project endpoints, Container Registry, and ACA resource names.

In [ ]:
output = utils.run(
    f'az deployment group show --name {deployment_name} -g {resource_group_name}',
    f'Retrieved deployment: {deployment_name}',
    f'Failed to retrieve deployment: {deployment_name}'
)

if output.success and output.json_data:
    log_analytics_id             = utils.get_deployment_output(output, 'logAnalyticsWorkspaceId',        'Log Analytics Id')
    apim_service_id              = utils.get_deployment_output(output, 'apimServiceId',                  'APIM Service Id')
    apim_resource_gateway_url    = utils.get_deployment_output(output, 'apimResourceGatewayURL',         'APIM Gateway URL')
    container_registry_name      = utils.get_deployment_output(output, 'containerRegistryName',          'Container Registry Name')
    container_registry_server    = utils.get_deployment_output(output, 'containerRegistryLoginServer',   'Container Registry Server')
    weather_mcp_aca_name         = utils.get_deployment_output(output, 'weatherMCPServerContainerAppName', 'Weather MCP ACA Name')
    weather_mcp_fqdn             = utils.get_deployment_output(output, 'weatherMCPServerFQDN',           'Weather MCP FQDN')
    hosted_agent_aca_name        = utils.get_deployment_output(output, 'hostedAgentContainerAppName',    'Hosted Agent ACA Name')
    hosted_agent_fqdn            = utils.get_deployment_output(output, 'hostedAgentFQDN',                'Hosted Agent FQDN')
    foundry_inference_endpoint   = utils.get_deployment_output(output, 'foundryInferenceProjectEndpoint','Foundry Inference Endpoint')
    foundry_agent_endpoint       = utils.get_deployment_output(output, 'foundryAgentProjectEndpoint',    'Foundry Agent Endpoint')
    weather_mcp_connection_id    = utils.get_deployment_output(output, 'weatherMCPConnectionId',         'Weather MCP Connection Id')

    apim_subscriptions = json.loads(
        utils.get_deployment_output(output, 'apimSubscriptions').replace("\'", '"')
    )
    for sub in apim_subscriptions:
        utils.print_info(f"Subscription Name: {sub['name']}")
        utils.print_info(f"Subscription Key:  ****{sub['key'][-4:]}")
    api_key = apim_subscriptions[0].get('key')


<a id='build-weather'></a>
### 🐳 Build and deploy the Weather MCP Server to ACA

The Weather MCP server code is in [`../../shared/mcp-servers/weather/http/`](../../shared/mcp-servers/weather/http/).
We build the Docker image and push it to the Azure Container Registry, then update the ACA.

In [ ]:
build=build+1

weather_image_tag = f'weather-mcp:{build}'
weather_mcp_src = '../../shared/mcp-servers/weather/http'

utils.print_info(f'Building Weather MCP image: {weather_image_tag}')

# Build and push the Weather MCP Server image
output = utils.run(
    f'az acr build --registry {container_registry_name} --image {weather_image_tag} {weather_mcp_src}',
    'Weather MCP image built and pushed',
    'Failed to build and push Weather MCP image'
)

In [ ]:
# Update the Weather MCP ACA with the new image
output = utils.run(
    f'az containerapp update --name {weather_mcp_aca_name} --resource-group {resource_group_name} --image {container_registry_name}.azurecr.io/{weather_image_tag}',
    f'Weather MCP ACA updated: {weather_mcp_aca_name}',
    f'Failed to update Weather MCP ACA: {weather_mcp_aca_name}'
)

utils.print_info(f'Weather MCP server available at: https://{weather_mcp_fqdn}/mcp')

<a id='local'></a>
### 🧪 Test the hosted agent locally

Before deploying to ACA, run the agent locally to verify it works correctly.
This installs the agent's dependencies and starts it on **http://localhost:8088**.

> **Note**: Stop the local server (`Ctrl+C` in the terminal) before proceeding to the deployment steps.

In [ ]:
# Write a .env file for the local agent run
env_content = f"""AZURE_OPENAI_ENDPOINT={apim_resource_gateway_url}/{inference_api_path}
AZURE_OPENAI_CHAT_DEPLOYMENT_NAME={models_config[0]['name']}
AZURE_OPENAI_KEY={api_key}
AZURE_AI_PROJECT_ENDPOINT={foundry_agent_endpoint}
WEATHER_MCP_CONNECTION_ID={weather_mcp_connection_id}
"""
with open('src/agent/.env', 'w') as f:
    f.write(env_content)

utils.print_ok('.env written to src/agent/.env')
utils.print_info('To run the agent locally, execute the following in a terminal:')
utils.print_info('  cd src/agent')
utils.print_info('  pip install -r requirements.txt')
utils.print_info('  python main.py')


In [ ]:
# Install the agent dependencies
output = utils.run(
    'pip install -r src/agent/requirements.txt -q',
    'Agent dependencies installed',
    'Failed to install agent dependencies'
)

In [ ]:
# Test via the Azure AI Foundry Responses API using the agent's localhost endpoint
# Testing both via the OpenAI Responses API client and direct HTTP call to /responses endpoint to show that both work with the hosted agent

import requests
from azure.identity import DefaultAzureCredential
from openai import responses, OpenAI

agent_url = f'http://localhost:8088/'
query = 'What is the weather like in London right now?'
utils.print_message(f'Q: {query}')

# The hosted agent responds to Responses API calls at /responses
test_payload = {
    'input': query
}

client=OpenAI(
    base_url=agent_url,
    api_key='xxxx'              ### Since this is a local agent endpoint, the API key is not validated, so we can use any string here.
)

response=client.responses.create(
    model="hosted_agent_localhost",          ### any arbitrary string to help identify the agent in the logs
    input=test_payload['input']
)

utils.print_ok(f'OpenAI Responses API A: {response.output_text}\n')

response = requests.post(
    url=f'{agent_url}/responses',
    headers={
        'Content-Type': 'application/json'
    },
    json=test_payload,
    timeout=60
)

if response.status_code == 200:
    utils.print_ok('Agent responded successfully via direct HTTP call')
    result = response.json()
    print(json.dumps(result, indent=2))
else:
    utils.print_error(f'Agent returned HTTP {response.status_code}: {response.text}')

<a id='build-agent'></a>
### 🐳 Build and deploy the Hosted Agent to ACA

Build the hosted agent Docker image from `src/agent/` and deploy it to the ACA.
The ACA is already configured with the required environment variables (APIM endpoint, Foundry project endpoint).
The Weather MCP connection ID is passed via the `.env` file during local testing;
after deployment it must be set as an ACA environment variable.


In [ ]:
build=build+2

agent_image_tag = f'hosted-agent:{build}'
agent_src_path = 'src/agent'

utils.print_info(f'Building Hosted Agent image: {agent_image_tag}')

output = utils.run(
    f'az acr build --registry {container_registry_name} --image {agent_image_tag} {agent_src_path}',
    'Hosted Agent image built and pushed',
    'Failed to build and push Hosted Agent image'
)

In [ ]:
# Update the Hosted Agent ACA with the new image and inject connection ID env vars
output = utils.run(
    f'az containerapp update '
    f'--name {hosted_agent_aca_name} '
    f'--resource-group {resource_group_name} '
    f'--image {container_registry_name}.azurecr.io/{agent_image_tag} '
    f'--set-env-vars '
    f'WEATHER_MCP_CONNECTION_ID="{weather_mcp_connection_id}"',
    f'Hosted Agent ACA updated: {hosted_agent_aca_name}',
    f'Failed to update Hosted Agent ACA: {hosted_agent_aca_name}'
)

utils.print_info(f'Hosted Agent available at: https://{hosted_agent_fqdn}')


<a id='test-aca'></a>
### 🧪 Test the hosted agent running in ACA

Send a test message to the deployed hosted agent using the [Azure AI Foundry Responses API](https://learn.microsoft.com/azure/ai-foundry/agents/overview).

> **Tip**: Use the [tracing tool](../../tools/tracing.ipynb) to monitor token usage and APIM policy behavior.

In [ ]:
# Test via the Azure AI Foundry Responses API using the agent's ACA endpoint
# using HTTP request to /responses endpoint
import requests

agent_url = f"https://{hosted_agent_fqdn}"
query = 'What is the weather like in London right now?'

# The hosted agent responds to Responses API calls at /responses
test_payload = {
    ##### you can specify any string as the 'model' in the payload, or in client response.create() call, to help identify the agent in the logs. 
    ##### This is especially useful if you have multiple agents deployed and want to differentiate them in the logs.
    # 'model': "hosted_agent_12.1",    ### any arbitrary string to help identify the agent in the logs
    'input': query
}

response = requests.post(
    url=f'{agent_url}/responses',
    headers={
        'Content-Type': 'application/json'
    },
    json=test_payload,
    timeout=60
)

if response.status_code == 200:
    utils.print_ok('Agent responded successfully via direct HTTP call')
    result = response.json()
    print(json.dumps(result, indent=2))
else:
    utils.print_error(f'Agent returned HTTP {response.status_code}: {response.text}')

In [ ]:
# Test via the Azure AI Foundry Responses API using the agent's ACA endpoint
# using OpenAI Responses API client
from azure.identity import DefaultAzureCredential
from openai import responses, OpenAI

agent_url = f"https://{hosted_agent_fqdn}"
query = 'What is the weather like in Lisbon and London right now?'

utils.print_message(f'Q: {query}')

# The hosted agent responds to Responses API calls at /responses
test_payload = {
    ##### you can specify any string as the 'model' in the payload, or in client response.create() call, to help identify the agent in the logs. 
    ##### This is especially useful if you have multiple agents deployed and want to differentiate them in the logs.
    # 'model': "hosted_agent_12.1",    ### any arbitrary string to help identify the agent in the logs
    'input': query
}

client=OpenAI(
    base_url=agent_url,
    api_key="xxxx"              ### Since this is a custom agent endpoint, the API key is not validated, so we can use any string here.
)

response=client.responses.create(
    ##### you can specify any string as the 'model' in the payload, or in client response.create() call, to help identify the agent in the logs. 
    ##### This is especially useful if you have multiple agents deployed and want to differentiate them in the logs.
    model="hosted_agent_12.0",          ### any arbitrary string to help identify the agent in the logs
    input=test_payload['input']
)

utils.print_ok(f'A: {response.output_text}')

<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.